# Qualifying Session
- **Q1**: Bottom 14 from Practice → top 2 advance to Q2
- **Q2**: Top 10 from Practice + 2 from Q1 → determines starting grid (P1–P12)
- Session time: 15 min (900s) → laps = floor(900 / base_lap_time) − 1
- Push mode: last 3 laps

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.insert(0, '..')
from src.engine import circuit_weights, perf_score_quali, simulate_quali_lap, fmt_lap, fmt_gap
from src.loader import load_riders, load_circuits, parse_practice_md

RAW       = Path('../data/raw')
PUSH_LAPS = 3

In [ ]:
df       = load_riders(RAW)
circuits = load_circuits(RAW)

CIRCUIT_IDX = 0
circuit    = circuits.iloc[CIRCUIT_IDX]
country    = circuit['country']
report_dir = Path('../report') / country

total_laps    = int(900 // circuit['base_lap_time']) - 1
push_from_lap = total_laps - PUSH_LAPS + 1

print(f"Circuit    : {circuit['circuit_name']} ({country})")
print(f"Base lap   : {circuit['base_lap_time']}s")
print(f"Total laps : {total_laps}  |  Push mode from lap: {push_from_lap}")

practice_df = parse_practice_md(report_dir)
top10_names = set(practice_df[practice_df['position'] <= 10]['name'])
print(f"\nTop 10 from Practice (→ Q2 direct): {len(top10_names)} riders")
print(f"Q1 riders: {len(df) - len(top10_names)}")

# ── Session runner ────────────────────────────────────────────────────────────
w_spd, w_cor, w_brk = circuit_weights(circuit)
base_time = float(circuit['base_lap_time'])

def run_session(riders, label):
    results = []
    for _, rider in riders.iterrows():
        score = perf_score_quali(rider, w_spd, w_cor, w_brk)
        lap_times = []
        for lap_num in range(1, total_laps + 1):
            t = simulate_quali_lap(rider, base_time, lap_num, score, lap_num >= push_from_lap)
            if t is not None:
                lap_times.append(t)
        if lap_times:
            results.append({'bike_number': int(rider['bike_number']), 'name': rider['name'],
                            'team': rider['team'], 'manufacturer': rider['manufacturer'],
                            'best_lap_sec': min(lap_times), 'best_lap': fmt_lap(min(lap_times))})
    res = pd.DataFrame(results).sort_values('best_lap_sec').reset_index(drop=True)
    res['gap']     = res['best_lap_sec'] - res['best_lap_sec'].iloc[0]
    res['gap_fmt'] = res['gap'].apply(fmt_gap)
    res.index += 1
    return res

print("Session runner ready.")

In [10]:
# ── Q1 Session ───────────────────────────────────────────────────────────────

q1_riders = df[~df['name'].isin(top10_names)].copy()
q1_class  = run_session(q1_riders, 'QUALIFYING 1')

# ── Q1 Classification ────────────────────────────────────────────────────────
print(f"\n{'─'*68}")
print( '  Q1 — CLASSIFICATION')
print(f"{'─'*68}")
print(f"  {'P':<4} {'#':<5} {'RIDER':<24} {'TEAM':<26} {'BEST LAP':>10} {'GAP':>8}")
print(f"  {'─'*66}")
for pos, row in q1_class.iterrows():
    advance = '  ★ ADVANCES TO Q2' if pos <= 2 else ''
    print(f"  P{pos:<3} #{row['bike_number']:<4} {row['name']:<24} {row['team']:<26} {row['best_lap']:>10} {row['gap_fmt']:>8}{advance}")

q2_advance_names = set(q1_class.head(2)['name'])
q1_non_q2        = q1_class.iloc[2:].copy()

print(f"\n{'='*68}")
print(f"  ★ Advancing to Q2: {', '.join(q2_advance_names)}")
print(f"{'='*68}")

  QUALIFYING 1
  LIAM HENDERSON CIRCUIT  —  AUSTRALIA
  Laps: 9  |  Push mode: last 3 laps (lap 7+)

  #10  Victor Burgos  |  Inferno Factory  [SATELLITE]
    LAP         TIME         BEST
  ───────────────────────────────────
  Lap  1    01:29.690    01:29.690  ◄ PB
  Lap  2    01:28.873    01:28.873  ◄ PB
  Lap  3    01:29.148    01:28.873
  Lap  4    01:29.053    01:28.873
  Lap  5    01:29.031    01:28.873
  Lap  6    01:29.054    01:28.873
  Lap  7    01:28.544    01:28.544  ◄ PB  [PUSH]
  Lap  8    01:28.947    01:28.544  [PUSH]
  Lap  9    01:28.245    01:28.245  ◄ PB  [PUSH]

  #12  Francesco Carelli  |  Phoenix Motorsport  [SATELLITE]
    LAP         TIME         BEST
  ───────────────────────────────────
  Lap  1    01:29.846    01:29.846  ◄ PB
  Lap  2    01:29.349    01:29.349  ◄ PB
  Lap  3    01:29.319    01:29.319  ◄ PB
  Lap  4    01:29.354    01:29.319
  Lap  5    01:29.180    01:29.180  ◄ PB
  Lap  6    01:29.358    01:29.180
  Lap  7    01:28.617    01:28.617  ◄ PB  

In [11]:
# ── Q2 Session ───────────────────────────────────────────────────────────────

q2_riders = pd.concat([
    df[df['name'].isin(top10_names)],
    df[df['name'].isin(q2_advance_names)]
], ignore_index=True)

q2_class = run_session(q2_riders, 'QUALIFYING 2')

# ── Q2 Classification ────────────────────────────────────────────────────────
print(f"\n{'─'*82}")
print( '  Q2 — CLASSIFICATION  (= STARTING GRID P1–P12)')
print(f"{'─'*82}")
print(f"  {'P':<4} {'#':<5} {'RIDER':<24} {'TEAM':<26} {'MANUFACTURER':<14} {'BEST LAP':>10} {'GAP':>8}")
print(f"  {'─'*78}")
for pos, row in q2_class.iterrows():
    pole = '  ◀ POLE' if pos == 1 else ''
    print(f"  P{pos:<3} #{row['bike_number']:<4} {row['name']:<24} {row['team']:<26} {row['manufacturer']:<14} {row['best_lap']:>10} {row['gap_fmt']:>8}{pole}")
print(f"{'─'*82}")

  QUALIFYING 2
  LIAM HENDERSON CIRCUIT  —  AUSTRALIA
  Laps: 9  |  Push mode: last 3 laps (lap 7+)

  #07  Petros Georgiou  |  Honda Factory Racing  [FACTORY]
    LAP         TIME         BEST
  ───────────────────────────────────
  Lap  1    01:28.919    01:28.919  ◄ PB
  Lap  2    01:28.424    01:28.424  ◄ PB
  Lap  3    01:28.671    01:28.424
  Lap  4    01:28.535    01:28.424
  Lap  5    01:28.520    01:28.424
  Lap  6    01:28.614    01:28.424
  Lap  7    01:28.719    01:28.424  [PUSH]
  Lap  8    01:28.637    01:28.424  [PUSH]
  Lap  9    01:28.377    01:28.377  ◄ PB  [PUSH]

  #19  Matteo Esposito  |  Honda Factory Racing  [FACTORY]
    LAP         TIME         BEST
  ───────────────────────────────────
  Lap  1    01:28.799    01:28.799  ◄ PB
  Lap  2    01:28.316    01:28.316  ◄ PB
  Lap  3    01:28.306    01:28.306  ◄ PB
  Lap  4    01:28.321    01:28.306
  Lap  5    01:28.301    01:28.301  ◄ PB
  Lap  6    01:28.289    01:28.289  ◄ PB
  Lap  7    01:27.786    01:27.786  ◄ P

In [12]:
# ── Export Q1.md / Q2.md / grid.md ──────────────────────────────────────────

title = f"{circuit['circuit_name']} - GRAND PRIX OF {country.upper()}"

def md_table(class_df):
    header = '| P | # | RIDER | TEAM | MANUFACTURER | BEST LAP | GAP |'
    sep    = '|---|---|-------|------|--------------|----------|-----|'
    rows   = [
        f"| P{pos} | #{r['bike_number']} | {r['name']} | {r['team']} | {r['manufacturer']} | {r['best_lap']} | {r['gap_fmt']} |"
        for pos, r in class_df.iterrows()
    ]
    return f"{header}\n{sep}\n" + "\n".join(rows)

# Q1.md
(report_dir / 'Q1.md').write_text(
    f"# {title}\n\n## Qualifying 1 - Classification\n\n{md_table(q1_class)}\n",
    encoding='utf-8'
)

# Q2.md
(report_dir / 'Q2.md').write_text(
    f"# {title}\n\n## Qualifying 2 - Classification\n\n{md_table(q2_class)}\n",
    encoding='utf-8'
)

# grid.md — Q2 results (P1–P12) | separator | Q1 non-qualifiers (P13–P24)
q1_nq = q1_non_q2.reset_index(drop=True)
q1_nq.index = range(13, 13 + len(q1_nq))
q1_nq['gap']     = q1_nq['best_lap_sec'] - q1_nq['best_lap_sec'].iloc[0]
q1_nq['gap_fmt'] = q1_nq['gap'].apply(fmt_gap)

grid_md = (
    f"# {title}\n\n"
    f"## Starting Grid\n\n"
    f"### Q2 Qualifiers\n\n{md_table(q2_class)}\n\n"
    f"---\n\n"
    f"### Q1 (Non-Qualifiers)\n\n{md_table(q1_nq)}\n"
)
(report_dir / 'grid.md').write_text(grid_md, encoding='utf-8')

print(f"Saved: Q1.md | Q2.md | grid.md  →  {report_dir}")

Saved: Q1.md | Q2.md | grid.md  →  ..\report\Australia
